### Collections Model to Identify Potential Defaulters

In [2]:
#Import Libraries
import seaborn as sns
import datetime
import pandas as pd
import numpy as np
import warnings
import pyodbc
import glob
import os
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.4f' % x)
pd.set_option('display.max_columns', None)

In [3]:
#Define the Model Periods

#Obtain the Collections Month
CMonth = datetime.datetime.now()

Collections_Year =CMonth.year
Collections_Month = CMonth.strftime('%b')
Collections_Month_No =f'{CMonth.month}'
Sales_Month = (CMonth - datetime.timedelta(days=30)).strftime('%b')#.upper()

Collections_Month_Year = f'{Collections_Month}_2024'
Month_Year = f'{Sales_Month}_2024'

#Importing the Active_DB  -----  update every month--Finance Folder
db = pd.read_csv(r"data\Collections\Collections Model\Support Files\Active_Db\Active_db_MAY.csv")
Colsn = ['SP_POINT_NO', 'REGION', 'OFFICE_NAME', 'SERVICE_NO','TARIFF_DESC', 'TARIFF_CLASS', 'SERVICE_STATUS']
db =  db.apply(lambda x: x.str.strip() if isinstance(x,str) else x)
db = db[Colsn]

In [4]:
Collections_Month

'Feb'

In [5]:
Collections_Month_No

'2'

In [6]:
Sales_Month

'Jan'

### Model Support Files

In [7]:
Collections_Month_Year = 'Dec_2024'

In [8]:
#GOU Accounts
GoU = pd.read_csv(r"data\Collections\Collections Model\Support Files\GOU Accounts\GOU_Accounts.csv",dtype='unicode')

#Model Collection Files - Using pre-merged Service_Points_Attached files
Collection_Files = glob.glob(fr"data\Collections\Collections Model\Collections_Serverfiles\Apr_2024\Service_Points_Attached\*.csv")

#Import Database With Service PointsSupport - needed for later merges
Service_Points = pd.read_csv(r"data\Collections\Collections Model\Support Files\Database of Service Points\All_Service_Points_DataBase.csv", dtype='unicode', encoding='latin-1')
Service_Points = Service_Points[['CONTRACT_NO','SP_POINT_NO']]

Columns = ['REGION', 'DISTRICT', 'CUSTOMER_NAME', 'TARIFF', 'TARIFF_CLASS', 'CONTRACT_NO', 'CONTRACT_STATUS', 'SP_POINT_NO', 'BILLED_AMOUNT', 'COLLECTED_AMOUNT']
Service_Points_attached = []
for File in Collection_Files:
    df = pd.read_csv(File, low_memory=False, encoding='latin-1')
    df.columns = [x.strip() for x in df.columns]
    df = df[Columns]
    df = df.dropna(subset=['DISTRICT','CONTRACT_NO']).reset_index(drop=True)
    df['CONTRACT_NO'] = df['CONTRACT_NO'].astype(int).astype(str)
    df  = df[~df['CONTRACT_STATUS'].isin(['INACTIVE.','INACTIVE WITH BALANCE.', 'INACTIVATION IN PROCESS.'])]
    df = df.drop_duplicates()
    df = df[~pd.isna(df['SP_POINT_NO'])]
    Service_Points_attached.append(df)

# Load Customer Names for later merges
Names_File = glob.glob(r"data\Collections\Collections Model\Support Files\Customer Names\*.csv")
Names = pd.concat([pd.read_csv(File, dtype='unicode', encoding='latin-1') for File in Names_File], ignore_index=True, axis=0)

In [9]:
Collections_Month_No

'2'

In [10]:
Collections_Year

2026

In [11]:
#Aged Debt Files
#Dont Delete - local data folder structure
Cols = ['SP_POINT_NO','CONTRACT_NO','ZERO_TO_THIRTY_DAYS', 'TOTAL_DEBT']

#Pick the Updated Debt File
try:
    Debt_Files = glob.glob(fr"data\Collections\Collections Daily\Consolidated Files\Daily_Collections\{Collections_Year}\{Collections_Month_No}\*.csv")
    Debt_Files_Filtered = [file for file in Debt_Files if 'consolidated' in file.lower()]
    Debt = pd.concat([pd.read_csv(File, usecols=Cols, dtype='unicode', encoding='latin-1') for File in Debt_Files_Filtered], ignore_index=True)       
    Debt.to_csv(fr"data\Collections\Collections Model\Support Files\Aged Debt\Aged_Debt_{Collections_Month}_{Collections_Year}.csv",index=False)
except:
    print('Error: Currently Not Connected to Server Files')
    Debt = pd.read_csv(fr"data\Collections\Collections Model\Support Files\Aged Debt\Aged_Debt_{Collections_Month}_{Collections_Year}.csv", dtype='unicode')
    print(f'Correction: Picked Last Exported Debt File - {Collections_Month}_{Collections_Year}')

# Obtaining a list of CSV files Concatenating the CSV files into a single DataFrame
Names_File = glob.glob(r"data\Collections\Collections Model\Support Files\Customer Names\*.csv")
Names = pd.concat([pd.read_csv(File, dtype='unicode', encoding='latin-1') for File in Names_File], ignore_index=True, axis=0)

#import Special Accounts
Special_Accounts_List = pd.read_csv(r"data\Collections\Collections Model\Support Files\Special Accounts\Special_Accounts_List.csv", dtype='unicode', encoding='latin-1')

Error: Currently Not Connected to Server Files
Correction: Picked Last Exported Debt File - Feb_2026


In [12]:
#Collection Reports File Processing.
dfs = []
for File in Service_Points_attached:
    df = File.dropna(subset=['REGION'], axis=0)
    df[['BILLED_AMOUNT', 'COLLECTED_AMOUNT']] = df[['BILLED_AMOUNT', 'COLLECTED_AMOUNT']].astype(float)
    df['Rate'] = df.apply(lambda row: 1 if row['BILLED_AMOUNT'] == 0 else (row['COLLECTED_AMOUNT'] / row['BILLED_AMOUNT']) if row['COLLECTED_AMOUNT'] != 0 else 0, axis=1)
    df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')
    df = df.drop_duplicates(subset='SP_POINT_NO')
    df = df[df['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
    df[['CONTRACT_NO','SP_POINT_NO']] = df[['CONTRACT_NO','SP_POINT_NO']].astype(int).astype(str)
    df.fillna(1, inplace=True)
    df['Rate'] = df['Rate'].astype(float)
    dfs.append(df)
    
#Assign File Names to Dataframes
[df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025] = dfs

#Define the lowest Years
df_names = []
for Yr in range(2017,2026):
    df_Yr = f'df_{Yr}'
    df_names.append(df_Yr)
# Create lists to store data
data = []
# Iterate through the DataFrames and their names
for df, df_name in zip(dfs, df_names):
    Billed_Sum = df['BILLED_AMOUNT'].sum(axis=0)
    Collected_Sum = df['COLLECTED_AMOUNT'].sum(axis=0)
    data.append([df_name, Billed_Sum, Collected_Sum])
# Create a new DataFrame
result_df = pd.DataFrame(data, columns=['Month', 'Billed_Sum', 'Collected_Sum'])
# Print the resulting DataFrame
result_df['Rate'] = (result_df['Collected_Sum']/result_df['Billed_Sum'])
result_df

,Month,Billed_Sum,Collected_Sum,Rate
0,df_2017,112579157205.2800,102890692063.8400,0.9139
1,df_2018,115300737828.6600,111773227700.8300,0.9694
2,df_2019,125278450489.9800,120587768891.4600,0.9626
3,df_2020,117914703748.8500,79886550324.4300,0.6775
4,df_2021,119947217324.0600,111403674760.3000,0.9288
5,df_2022,121272756714.8900,114000546176.6400,0.9400
6,df_2023,135769114960.8000,125583134097.7100,0.9250
7,df_2024,0.0000,0.0000,NaN
8,df_2025,0.0000,0.0000,NaN


In [13]:
#Average Collection Rate
(result_df['Rate'].iloc[:-1].mean())*100

np.float64(90.24545241291545)

### Workings

In [14]:
#Workings
Ws = db.copy()
Ws = Ws[Ws['TARIFF_CLASS']!='OTHERS']
Ws = Ws.rename(columns={'SERVICE_NO':'CONTRACT_NO'})
Ws[['CONTRACT_NO','SP_POINT_NO']] = Ws[['CONTRACT_NO','SP_POINT_NO']].astype(int).astype(str)
Ws = Ws.drop_duplicates(subset='CONTRACT_NO')

#Compute and Attach Rates
Years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Year_1 = Years[0]
Year_2 = Years[-1]

Cols_2 = ['SP_POINT_NO','Rate']
for Year, df in zip(Years, dfs):
    Ws = pd.merge(Ws, dfs[Year-2017][Cols_2], on ='SP_POINT_NO', how='left')
    Ws = Ws.drop_duplicates(subset='CONTRACT_NO')
    Ws['Rate'].fillna(1, inplace=True)
    Ws['Rate'] = Ws['Rate'].astype(float)
    Ws.rename(columns={'Rate':f'Rate_{Year}'}, inplace=True)
    Ws.loc[:,f'Rate_{Year_1}':] = Ws.loc[:,f'Rate_{Year_1}':].fillna(1)
#Convert to Percentages
Ws.loc[:,f'Rate_{Year_1}':] = Ws.loc[:,f'Rate_{Year_1}':]*100

#Default Mappings
for Year in Years:
    Ws[f'Default_{Year}'] = Ws.apply(lambda row: 0 if row[f'Rate_{Year}'] == 0 else 2 if row[f'Rate_{Year}'] < 99 else 1, axis=1)
    
#Counts -  Defaults
Ws['Count'] = Ws.loc[:,f'Default_{Year_1}':f'Default_{Year_2}'].count(axis=1)
Ws['Counts_2'] = Ws.loc[:,f'Default_{Year_1}':f'Default_{Year_2}'].apply(lambda row: row.tolist().count(2), axis=1)
Ws['Counts_0'] =  Ws.loc[:,f'Default_{Year_1}':f'Default_{Year_2}'].apply(lambda row: row.tolist().count(0), axis=1)

#Probabilities
Ws['Prob_2'] = Ws.apply(lambda row: row['Counts_2']/row['Count'], axis=1)
Ws['Prob_0'] = Ws.apply(lambda row: row['Counts_0']/row['Count'], axis=1)

#Flat probability
Ws['Flat_Prob'] = Ws[['Prob_2','Prob_0']].sum(axis=1)

#Average Partial Probability
def Filter_and_Average(row):
    Filtered_Values = [x for x in row if 0 < x < 99]
    if len(Filtered_Values) > 0:
        average = sum(Filtered_Values) / len(Filtered_Values)
    else:
        average = 0       
    return average
# Apply the function to each row of the DataFrame
Ws['Avg_Partial'] = Ws.loc[:,f'Rate_{Year_1}':f'Rate_{Year_2}'].apply(Filter_and_Average , axis=1)

#Adjusted Probabilities
Ws['Weighted_Prob_2'] = ((100-Ws['Avg_Partial'])*Ws['Prob_2'])/100
Ws['Adj_Prob'] = Ws[['Prob_0','Weighted_Prob_2']].sum(axis=1)

#Default_Probability
Ws['Default_Probability'] = Ws['Adj_Prob'].apply(lambda x: 'Very High Probability of default' if x >= 0.5 else 'High Probability of default' if (0.3 <= x < 0.5) else 'Low Probability of default' if (0.01 <= x < 0.25) else 'Very Low Probability of default')

#Process Customer Names
Nms =  pd.concat([df_2024[['SP_POINT_NO','CUSTOMER_NAME']], Names[['SP_POINT_NO','CUSTOMER_NAME']]], axis=0, ignore_index=True)
Nms[['SP_POINT_NO','CUSTOMER_NAME']]  = Nms[['SP_POINT_NO','CUSTOMER_NAME']].astype(str)
Nms = Nms.drop_duplicates()

#Merge Customer Names
Ws =  pd.merge(Ws, Nms[['SP_POINT_NO','CUSTOMER_NAME']], on='SP_POINT_NO', how='left')
Ws = Ws.drop_duplicates(subset='CONTRACT_NO')
Ws['Sale_Cycle'] = Month_Year

#Attach Collected and Billed Amounts from Current Collections Report
Ws = pd.merge(Ws, df_2024[['SP_POINT_NO', 'BILLED_AMOUNT', 'COLLECTED_AMOUNT']], how='left', on='SP_POINT_NO')
Ws.fillna(0, inplace=True)
Ws = Ws.drop_duplicates(subset='CONTRACT_NO')

#Merge GOU Customers
Wsn = pd.merge(Ws,GoU[['CONTRACT_NO','Is_GOU?']], on='CONTRACT_NO', how='left')
Wsn['Is_GOU?'].fillna('NO',inplace=True)
Wsn = Wsn.drop_duplicates(subset='CONTRACT_NO')

In [ ]:
#Attach service Points to Credit Risk
#Import the Credit Risk Files
Crs = []
Crn_s = glob.glob(r"data\Collections\Collections Model\Support Files\Credit_Risk\*.csv")
Cr = pd.concat([pd.read_csv(cr, dtype='unicode', encoding='latin-1') for cr in Crn_s], ignore_index=True)
Cr = Cr.iloc[:,[0,-1]]
Cr.columns = ['CONTRACT_NO', 'CREDIT_RISK']
Cr = Cr.drop_duplicates(subset='CONTRACT_NO')
Cr = Cr.reset_index(drop=True)

#Assign Supply Points to the Credit Risk
Credit_Risk =  pd.merge(Cr, Service_Points[['SP_POINT_NO','CONTRACT_NO']], on='CONTRACT_NO',how='left')
Credit_Risk = Credit_Risk[Credit_Risk['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
Credit_Risk =  Credit_Risk.drop_duplicates(subset='CONTRACT_NO')

#Merge Workings and Credit Risk
Wsm =  pd.merge(Wsn, Credit_Risk[['SP_POINT_NO','CREDIT_RISK']], on='SP_POINT_NO', how='left')
Wsm['CREDIT_RISK'].fillna('0',inplace=True)
Wsm = Wsm.drop_duplicates(subset='CONTRACT_NO')

#print columns for debugging
print("Wsm columns:", Wsm.columns.tolist())

#Assign Credit Risk
def Assn_Credit_Risk(row):
    if row['CREDIT_RISK'] != '0':
        return row['CREDIT_RISK']
    elif (row['Default_Probability'] == 'Very High Probability of default') & (row['CREDIT_RISK'] == '0'): 
        return 'High Risk'
    elif (row['Default_Probability'] == 'High Probability of default') & (row['CREDIT_RISK'] == '0'):
        return 'Medium High Risk'
    else:
        return 'Low_Risk'
Wsm['CREDIT_RISK'] = Wsm.apply(Assn_Credit_Risk, axis=1)

#Merge Aged Debt Files and Workings
Debt[['ZERO_TO_THIRTY_DAYS', 'TOTAL_DEBT']] = Debt[['ZERO_TO_THIRTY_DAYS', 'TOTAL_DEBT']].astype(float)
Debt['Arrears_Abv_30days'] = Debt['TOTAL_DEBT'] - Debt['ZERO_TO_THIRTY_DAYS']

#Add Aged Debt to Workings
Wsm = pd.merge(Wsm, Debt[['CONTRACT_NO','Arrears_Abv_30days']], on='CONTRACT_NO', how='left')
Wsm['Arrears_Abv_30days'].fillna(0,inplace=True)
Wsm = Wsm[Wsm['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
Wsm = Wsm.drop_duplicates(subset='CONTRACT_NO')
Wsm['CUSTOMER_NAME'].fillna("0",inplace=True)

#Calculating the Index Scores
Wsm['CreditRiskScore'] = Wsm['CREDIT_RISK'].apply(lambda x: 1 if x=='low Risk' else 2 if x=='Medium Low Risk' else 3 if x=='Medium High Risk' else 4)
Wsm['DefaultProbScore'] = Wsm['Default_Probability'].apply(lambda x: 0 if x=='Very Low Probability of default' else 2 if x=='Low Probability of default' else 8 if x=='High Probability of default' else 10)
Wsm['Default_Score'] = (Wsm['CreditRiskScore']*Wsm['DefaultProbScore'])*2.5

#Calculating the Factor of Recency in the Customer's Payments
Recency_Scores = [8, 12]
Yearz = [Year_2-1, Year_2]

for Recence_Score,Year in zip(Recency_Scores, Yearz):
    Wsm[f'Rec_Score_{Year}'] = Wsm[f'Default_{Year}'].apply(lambda x: Recence_Score if x==1 else 0)
Wsm['Recency_Scores'] = Wsm.loc[:,f'Rec_Score_{Year_2-1}':f'Rec_Score_{Year_2}'].sum(axis=1)
Wsm['Final_Default_Score'] = Wsm['Default_Score'] - Wsm['Recency_Scores']

#Identify Special Accounts
Wsm = pd.merge(Wsm, Special_Accounts_List[['SP_POINT_NO','Arch_Accts']], on='SP_POINT_NO', how='left').fillna('No')
Wsm['Arch_Accts'] = Wsm['Arch_Accts'].fillna('No')
Wsm = Wsm.drop_duplicates(subset=['SP_POINT_NO'])

#Filters for Potential Defaulters
Filter1 = Wsm['Final_Default_Score']>=20
Filter2 = Wsm['Is_GOU?']=='NO'
Filter3 = Wsm['TARIFF_CLASS']!='OTHERS'
Filter4 = Wsm['BILLED_AMOUNT'] > 100000

#Select Columns for Potential Defaulters - only select columns that exist
available_cols = Wsm.columns.tolist()
desired_cols = ['SP_POINT_NO', 'REGION', 'OFFICE_NAME', 'CONTRACT_NO', 'CUSTOMER_NAME','TARIFF_DESC', 'TARIFF_CLASS', 'SERVICE_STATUS', 
'Is_GOU?','BILLED_AMOUNT', 'COLLECTED_AMOUNT','CREDIT_RISK', 'Arrears_Abv_30days','Adj_Prob','Default_Probability','Final_Default_Score','Arch_Accts']
Colsn = [col for col in desired_cols if col in available_cols]

#Filter Out the Potential defaulters
Pd = Wsm[Filter1 & Filter2 & Filter3 & Filter4][Colsn]

Pd.reset_index(drop=True,inplace=True)
Pd = Pd[Pd['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
Pd = Pd.drop_duplicates(subset='SP_POINT_NO')

# Format Adj_Prob if it exists
if 'Adj_Prob' in Pd.columns:
    Pd['Adj_Prob'] = Pd['Adj_Prob'].apply(lambda x: f"{x*100:.1f}%")

# Format the 'values' column with thousands separators
if 'BILLED_AMOUNT' in Pd.columns:
    Potental_Defaulters = Pd.sort_values(by='BILLED_AMOUNT', ascending=False)
    Potental_Defaulters = Potental_Defaulters.reset_index(drop=True)
    Potental_Defaulters[['BILLED_AMOUNT','COLLECTED_AMOUNT','Arrears_Abv_30days']] = Potental_Defaulters[['BILLED_AMOUNT','COLLECTED_AMOUNT','Arrears_Abv_30days']].applymap('{:,.0f}'.format)
else:
    Potental_Defaulters = Pd.copy()

#Export an Unfilterd File
try:
    Wsm.to_excel(fr"data\Collections\Collections Model\Model Exports\UnFiltered_{Collections_Month_Year}.xlsx",index=False)
    print(f"✓ Successfully exported UnFiltered_{Collections_Month_Year}.xlsx")
except PermissionError:
    print(f"⚠ Unable to save UnFiltered file - it may be open in another application. Please close it and try again.")

Potental_Defaulters = Potental_Defaulters.drop_duplicates()
try:
    Potental_Defaulters.to_csv(fr"data\Collections\Collections Model\Model Exports\Potential_Defaulters_{Collections_Month_Year}.csv", index=False)
    print(f"✓ Successfully exported Potential_Defaulters_{Collections_Month_Year}.csv")
except PermissionError:
    print(f"⚠ Unable to save CSV file - it may be open in Excel. Please close it and try again.")
    
Potental_Defaulters.head(5)

Wsm columns: ['SP_POINT_NO', 'REGION', 'OFFICE_NAME', 'CONTRACT_NO', 'TARIFF_DESC', 'TARIFF_CLASS', 'SERVICE_STATUS', 'Rate_2017', 'Rate_2018', 'Rate_2019', 'Rate_2020', 'Rate_2021', 'Rate_2022', 'Rate_2023', 'Rate_2024', 'Rate_2025', 'Default_2017', 'Default_2018', 'Default_2019', 'Default_2020', 'Default_2021', 'Default_2022', 'Default_2023', 'Default_2024', 'Default_2025', 'Count', 'Counts_2', 'Counts_0', 'Prob_2', 'Prob_0', 'Flat_Prob', 'Avg_Partial', 'Weighted_Prob_2', 'Adj_Prob', 'Default_Probability', 'CUSTOMER_NAME', 'Sale_Cycle', 'BILLED_AMOUNT', 'COLLECTED_AMOUNT', 'Is_GOU?', 'CREDIT_RISK']


PermissionError: [Errno 13] Permission denied: 'data\\Collections\\Collections Model\\Model Exports\\Potential_Defaulters_Dec_2024.csv'